# Profiling Academic Trajectories: A ML Exploration of Student Performance

Jairik "JJ" McCauley

## Introduction

The Higher Education Students Performance Evaluation dataset [1] contains a mix of demographic, socioeconomic, academic, and behavioral attributes collected from students. Due to the wide array of features (32 in total), there is an immense amount of flexibility for analysis and exploration. Understanding what influences student performance has implications for advising, resource allocation, and instructional design within higher education.

In this project, I aim to identify different clusters of students, apply classification models to predict GPA categories, and gain insight into how individual features may impact perceived preparation and interest in a chosen field. These analyses enable exploration of not only predictive performance, but also broader patterns that may support student success and institutional planning.

[1]: Yilmaz, N. & Şekeroğlu, B. (2019). Higher Education Students Performance Evaluation [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C51G82


## Data Exploration

Within the following sections, we will be understanding the features of the dataset, describing it using scientific methods, wrangling it for clean analysis, and formulating clear research questions that we can answer.

### Understanding the Dataset

Prior to diving into this large dataset, it is important to understand the overall structure with each feature and type in the dataset:

| #  | Feature                              | Type                                  |
| -- | ------------------------------------ | ------------------------------------- |
| 1  | Student Age                          | Categorical (ordinal)                 |
| 2  | Sex                                  | Binary                                |
| 3  | Graduated High-School Type           | Categorical                           |
| 4  | Scholarship Type                     | Categorical (ordinal)                 |
| 5  | Additional Work                      | Binary                                |
| 6  | Artistic/Sports Activity             | Binary                                |
| 7  | Has a Partner                        | Binary                                |
| 8  | Total Salary                         | Categorical (ordinal)                 |
| 9  | Transportation Mode                  | Categorical                           |
| 10 | Accommodation Type                   | Categorical                           |
| 11 | Mother’s Education                   | Categorical (ordinal)                 |
| 12 | Father’s Education                   | Categorical (ordinal)                 |
| 13 | Number of Siblings                   | Categorical (ordinal)                 |
| 14 | Parental Status                      | Categorical                           |
| 15 | Mother’s Occupation                  | Categorical                           |
| 16 | Father’s Occupation                  | Categorical                           |
| 17 | Weekly Study Hours                   | Categorical (ordinal)                 |
| 18 | Reading Frequency (Non-Scientific)   | Categorical (ordinal)                 |
| 19 | Reading Frequency (Scientific)       | Categorical (ordinal)                 |
| 20 | Attends Seminars/Conferences         | Binary                                |
| 21 | Impact of Projects/Activities        | Categorical                           |
| 22 | Class Attendance                     | Categorical (ordinal)                 |
| 23 | Midterm Prep Method                  | Categorical                           |
| 24 | Midterm Prep Timing                  | Categorical                           |
| 25 | Takes Notes in Class                 | Categorical (ordinal)                 |
| 26 | Listens in Class                     | Categorical (ordinal)                 |
| 27 | Discussion Improves Interest/Success | Categorical (ordinal)                 |
| 28 | Flip Classroom Usefulness            | Categorical                           |
| 29 | Cumulative GPA (Last Semester)       | Categorical (ordinal)                 |
| 30 | Expected Cumulative GPA (Graduation) | Categorical (ordinal)                 |
| 31 | Course ID                            | Identifier / Categorical              |
| 32 | Output Grade (Letter Grade)          | Target Variable (Ordinal Categorical) |

From the dataset documentation, we can see that features 1-10 are personal questions, 11-16 are family questions, and 17-32 include educational habits. Additionally, it is important to note that this is a relatively small dataset, containing only around 145 instances. Therefore, generalization will be an important consideration throughout this analysis and exploration.


### Describing the Dataset

Firstly, we will load in the data using UCI's ML library.

In [2]:
uv pip install ucimlrepo  # Install the ucimlrepo package, if not already installed (not a standard library)

Using Python 3.12.12 environment at: /mnt/c/Users/jairi/OneDrive/Desktop/Repos/Data-Science-Fundementals/.venv
Resolved 8 packages in 890ms                                         ⠋ Resolving dependencies...                                                     
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/7.85 KiB            
⠙ Preparing packages... (0/1)---------- 7.85 KiB/7.85 KiB           
Prepared 1 package in 142ms                                                       
░░░░░░░░░░░░░░░░░░░░ [0/1] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in

In [3]:
from ucimlrepo import fetch_ucirepo  # Fetching the dataset
import pandas as pd  # Data manipulation

# Fetch this dataset from the UCI Machine Learning Repository
higher_education_students_performance = fetch_ucirepo(id=856)

# Extract the full data, metadata, and variable information
hesp_data_raw = higher_education_students_performance.data
hesp_metadata = higher_education_students_performance.metadata
hesp_variable_info = higher_education_students_performance.variable_info

# Convert the raw data into a pandas DataFrame for easier manipulation
hesp_df_raw = pd.concat([hesp_data_raw.features, hesp_data_raw.targets], axis=1)
type(hesp_df_raw)

pandas.core.frame.DataFrame

Now, we can use this dataframe to gain actionable insights on the dataframe. Firstly, we can view the metadata and variable info of the dataset, as provided by UCI.

In [4]:
from pprint import pprint  # For pretty-printing

# Display metadata
print("=== Dataset Metadata ===")
pprint(hesp_metadata)

print("\n=== Variable Information ===")
print(hesp_variable_info)

=== Dataset Metadata ===
{'abstract': 'The data was collected from the Faculty of Engineering and '
             'Faculty of Educational Sciences students in 2019. The purpose is '
             "to predict students' end-of-term performances using ML "
             'techniques.',
 'additional_info': {'citation': 'YÄ±lmaz N., Sekeroglu B. (2020) Student '
                                 'Performance Classification Using Artificial '
                                 'Intelligence Techniques. In: Aliev R., '
                                 'Kacprzyk J., Pedrycz W., Jamshidi M., '
                                 'Babanli M., Sadikoglu F. (eds) 10th '
                                 'International Conference on Theory and '
                                 'Application of Soft Computing, Computing '
                                 'with Words and Perceptions - ICSCCW-2019. '
                                 'ICSCCW 2019. Advances in Intelligent Systems '
                                

Additionally, we can confirm the shape and datatypes.

In [5]:
print(f"Shape: {hesp_df_raw.shape}")  # Display dataframe shape
pprint(hesp_df_raw.info())  # Display dataframe info

Shape: (145, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 32 columns):
 #   Column                                                             Non-Null Count  Dtype
---  ------                                                             --------------  -----
 0   Student Age                                                        145 non-null    int64
 1   Sex                                                                145 non-null    int64
 2   Graduated high-school type                                         145 non-null    int64
 3   Scholarship type                                                   145 non-null    int64
 4   Additional work                                                    145 non-null    int64
 5   Regular artistic or sports activity                                145 non-null    int64
 6   Do you have a partner                                              145 non-null    int64
 7   Total salary if available  

As we can see from the info description (and dataset documentation), almost all columns are categorical, but represented by an integer scale. The last observation that should be made regards missing values. Although the documentation states that there are no missing values, we can easily confirm this.

In [12]:
print(f"Null Values: {hesp_df_raw.isnull().sum().sum()}")  # Check for all null/missing values and print total count
print(f"NaN Values: {hesp_df_raw.isna().sum().sum()}")  # Check for all nan values and print total count
print(f"Empty String Values: {(hesp_df_raw == '').sum().sum()}")  # Check for all empty string values and print total count

Null Values: 0
NaN Values: 0
Empty String Values: 0


As we can see from these checks (and as the documentation confirms), there are no observable missing values.

### Data Wrangling

This dataset has conveniently came with numerical representations of categorical values and with no numerical values. Therefore, there is no large data wrangling that will be needed at this point.

Additionally, instead of re-mapping the numerically labeled data, it could be more beneficial to fully understand the original meaning of the labels for each column. This is clearly outlined in the dataset's documentation, and can be easily represented by the following table:

<!-- Large table describing the numerical meanings behind the labels for each column -->
| Column | Possible Labels | Label meanings |
|---|---|---|
| Student Age | 1, 2, 3 | Student’s age group: 1 = 18–21; 2 = 22–25; 3 = ≥26 years. |
| Sex | 1, 2 | Biological sex: 1 = female; 2 = male. |
| High School Type | 1, 2, 3 | Type of high school attended: 1 = private; 2 = public/state; 3 = other. |
| Scholarship Type | 1–5 | Financial aid level: 1 = none; 2 = 25%; 3 = 50%; 4 = 75%; 5 = full scholarship. |
| Additional Work | 1, 2 | Whether student works while studying: 1 = yes; 2 = no. |
| Artistic/Sports Activity | 1, 2 | Participation in extracurricular arts or athletics: 1 = yes; 2 = no. |
| Partnership Status | 1, 2 | Whether student currently has a partner: 1 = yes; 2 = no. |
| Monthly Salary (if working) | 1–5 | Monthly income bracket: 1 = $135–200; 2 = $201–270; 3 = $271–340; 4 = $341–410; 5 = >$410. |
| Transportation Mode | 1–4 | Means of commuting to school: 1 = bus; 2 = car/taxi; 3 = bicycle; 4 = other. |
| Accommodation Type | 1–4 | Living situation while studying: 1 = rental; 2 = dormitory; 3 = family home; 4 = other. |
| Mother’s Education | 1–6 | Highest qualification: 1 = primary; 2 = secondary; 3 = high school; 4 = university degree; 5 = MSc; 6 = PhD. |
| Father’s Education | 1–6 | Same encoding as mother’s education. |
| Number of Siblings | 1–5 | Number of siblings: 1 = one; 2 = two; 3 = three; 4 = four; 5 = five or more. |
| Parental Status | 1–3 | Family background: 1 = married parents; 2 = divorced; 3 = parental loss. |
| Mother’s Occupation | 1–6 | Employment category: 1 = retired; 2 = housewife; 3 = government employee; 4 = private sector; 5 = self-employed; 6 = other. |
| Father’s Occupation | 1–5 | Employment category: 1 = retired; 2 = government employee; 3 = private sector; 4 = self-employed; 5 = other. |
| Weekly Study Hours | 1–5 | Hours spent studying per week: 1 = none; 2 = < 5 hrs; 3 = 6–10 hrs; 4 = 11–20 hrs; 5 = > 20 hrs. |
| Recreational Reading | 1–3 | Frequency of reading non-academic texts: 1 = never; 2 = sometimes; 3 = often. |
| Academic Reading | 1–3 | Frequency of reading scientific/academic texts: 1 = never; 2 = sometimes; 3 = often. |
| Seminar/Conference Attendance | 1, 2 | Participation in academic seminars: 1 = yes; 2 = no. |
| Impact of Projects/Activities | 1–3 | Perceived effect of participation on academic success: 1 = positive; 2 = negative; 3 = neutral. |
| Course Attendance | 1–3 | How consistently student attends classes: 1 = always; 2 = sometimes; 3 = never. |
| Exam Preparation Style | 1–3 | Preparation approach for first midterm: 1 = individually; 2 = with peers; 3 = not applicable. |
| Exam Preparation Timing | 1–3 | Timing of preparation: 1 = only right before exams; 2 = regularly; 3 = never. |
| Note-Taking Frequency | 1–3 | How often student takes notes: 1 = never; 2 = sometimes; 3 = always. |
| Listening in Classes | 1–3 | In-class attentiveness: 1 = never; 2 = sometimes; 3 = always. |
| Effect of Discussion | 1–3 | Whether discussion enhances learning: 1 = never; 2 = sometimes; 3 = always. |
| Perceived Usefulness of Flipped Classroom | 1–3 | View of flipped-learning method: 1 = not useful; 2 = useful; 3 = not applicable. |
| Previous semester GPA (/4) | 1–5 | GPA category: 1 = <2.00; 2 = 2.00–2.49; 3 = 2.50–2.99; 4 = 3.00–3.49; 5 = >3.49. |
| Expected Graduation GPA (/4) | 1–5 | Expected GPA category, same encoding as above. |
| Course ID | text/integer | Identifier representing the specific course taken by student. |
| Final Course Grade (Target) | 0–7 | Letter-grade category: 0 = Fail; 1 = DD; 2 = DC; 3 = CC; 4 = CB; 5 = BB; 6 = BA; 7 = AA. |



Addiitonally, just as one last confirmation that the data was correcltly loaded, we can observe the head of the dataframe:

In [17]:
hesp_df_raw.head()  # Display the first 5 rows of the dataframe

,Student Age,Sex,Graduated high-school type,Scholarship type,Additional work,Regular artistic or sports activity,Do you have a partner,Total salary if available,Transportation to the university,Accomodation type in Cyprus,...,Preparation to midterm exams 1,Preparation to midterm exams 2,Taking notes in classes,Listening in classes,Discussion improves my interest and success in the course,Flip-classroom,Cumulative grade point average in the last semester (/4.00),Expected Cumulative grade point average in the graduation (/4.00),Course ID,OUTPUT Grade
0,2,2,3,3,1,2,2,1,1,1,...,1,1,3,2,1,2,1,1,1,1
1,2,2,3,3,1,2,2,1,1,1,...,1,1,3,2,3,2,2,3,1,1
2,2,2,2,3,2,2,2,2,4,2,...,1,1,2,2,1,1,2,2,1,1
3,1,1,1,3,1,2,1,2,1,2,...,1,2,3,2,2,1,3,2,1,1
4,2,2,1,3,2,2,1,3,1,4,...,2,1,2,2,2,1,2,2,1,1


### Defining Research Questions

To help clearly define our research questions, we can firstly develop a correlation matrix to understand the relationships among features. This will give us insight on potential relationships among features and aid in modeling decisions.

In [18]:
import plotly.express as px  # Interactive visualizations

# Calculate the correlation matrix
corr_matrix = hesp_df_raw.corr()

# Create the heatmap using Plotly
fig = px.imshow(corr_matrix,
                text_auto=".1f",
                aspect="auto",
                color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1,
                title='Correlation Matrix of Student Performance Features')

# Update layout for better readability
fig.update_layout(width=1000, height=1000)

fig.show()

## Research and Methods

Now that the dataset has been explored and understood, we can begin articulating our curiosities and using relevant ML methods to prove them.

Firstly, we can observe